###Gold — physical_lojas

KPIs de negócio gerados neste notebook (planilha "Squad 3 — Negócio/Técnica", aluno Luis Henrique, regras 4-10):

#	KPI |	Granularidade
4	Receita total por loja por ano	 | loja + ano

5	Crescimento YoY da receita por loja |	loja + ano

6	Número de transações por loja por mês |	loja + ano + mês

7	Enriquecimento IBGE (população e renda por cidade) |	loja

8	Ticket médio por loja por semestre |	loja + ano + semestre

9	Dicionário de dados Gold (documentado abaixo)	—

10	Validação: todas as lojas da Silver estão na Gold	—

Dicionário de dados Gold (KPI 9) — granularidade: 1 linha por loja por mês (colunas de ano/semestre são desnormalizadas via JOIN para facilitar consumo pelo time de BI no Looker, sem exigir jobs adicionais):

 - id_loja: identificador único da loja (PK).
 - nome_loja: nome da loja, padronizado na Silver.
 - cidade_loja: cidade da loja, padronizada na Silver.
 - estado_tratado: UF normalizada (2 letras), gerada na Silver.
 - populacao_cidade: população do município (Censo IBGE 2022).
 - renda_media_per_capita: renda domiciliar per capita média do município (Censo IBGE 2022, tabela Sidra 10295).
 - ano: ano de referência do KPI (4 dígitos).
 - mes: mês de referência do KPI (1-12), quando aplicável.
 - semestre: semestre de referência (1 ou 2), quando aplicável.
- receita_total: soma de valor_item_analitico dos itens da loja no período (usa apenas registros sem flag de FK inválida).
 - qtd_transacoes: contagem distinta de id_transacao no período.
 - ticket_medio: receita_total / qtd_transacoes no período.
 - receita_ano_anterior: receita do mesmo período no ano anterior (para cálculo de crescimento YoY).
 - crescimento_yoy_pct: variação percentual da receita YoY.

####Regras de filtragem aplicadas nesta camada:

 - Registros com flag_fk_invalido = true são excluídos (não é possível associar o item a uma loja/data sem a transação válida).
 - Registros com flag_quantidade_invalido = true ou flag_preco_invalido = true são excluídos (valor_item_analitico seria zero ou negativo, distorcendo os KPIs de receita).
 - Registros com flag_cnpj_invalido = true ou flag_uf_invalido = true permanecem (não afetam os KPIs de receita, mas afetam o enriquecimento IBGE — documentado).

O que este notebook faz:

 - Lê da Silver (squad3/silver/physical_lojas e squad3/silver/physical_itens_venda_caixa).
 - Lê o arquivo de municípios do IBGE (caminho definido por RAW_IBGE_MUNICIPIOS_PATH no 00_config) para enriquecimento.
 - Calcula os KPIs 4-8.
 - Valida que todas as lojas da Silver estão presentes na Gold (KPI 10).
 - Grava no SQL Server (squad3.gold_physical_lojas) via write_sql_table_typed() — única camada que grava no banco.
 - Grava também em Delta (squad3/gold/physical_lojas) para consumo pelos notebooks de Analysis.

In [0]:
%run "../utils/00_utils"

In [0]:
adls_options = get_adls_options()

# Modo de escrita da Gold:
#   "overwrite" -> reprocessamento completo
#   "append"    -> carga incremental (novos dados no raw)
# Altere para "append" em execuções normais de atualização de dados.
GOLD_WRITE_MODE = "overwrite"

In [0]:
from pyspark.sql.functions import (
    sum as spark_sum, count, countDistinct,
    avg, round as spark_round, year, month,
    quarter, when, lag, col, lit, expr,
    concat_ws, upper, trim
)
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

## Leitura da Silver — physical_lojas

In [0]:
df_lojas = read_delta(SILVER_LOJAS_PATH, adls_options)

print(f"Lojas na Silver: {df_lojas.count()}")
display(df_lojas.limit(5))

## Leitura da Silver — physical_itens_venda_caixa

Fonte dos dados de receita. Aplicamos aqui os filtros de qualidade
decididos na Gold (ver regras de filtragem no cabeçalho).

In [0]:
df_itens = (
    read_delta(SILVER_ITENS_VENDA_CAIXA_PATH, adls_options)
    .filter(col("flag_fk_invalido") == False)
    .filter(col("flag_quantidade_invalido") == False)
    .filter(col("flag_preco_invalido") == False)
)

print(f"Itens válidos para KPIs: {df_itens.count()}")

## Enriquecimento IBGE (KPI 7)

Lê o arquivo de municípios do IBGE salvo no container raw pelo
notebook `00_setup_ibge`. Faz JOIN com `cidade_loja` da Silver para
trazer a população do município de cada loja.

Se o arquivo ainda não existir no raw (antes de rodar o
`00_setup_ibge`), o bloco de enriquecimento é ignorado com aviso
— os demais KPIs são calculados normalmente.


####Enriquecimento IBGE automático (sem passo manual)

Chama 00_setup_ibge automaticamente a cada execução da Gold. O próprio notebook decide internamente se precisa buscar dados novos na API (dados com mais de 365 dias) ou se pode encerrar cedo sem custo de rede (dados ainda válidos) — ver a checagem de atualidade no início daquele notebook.



In [0]:
%run "../utils/00_setup_ibge"

In [0]:
IBGE_BRONZE_PATH = f"{BRONZE_BASE_PATH}ibge_municipios"

try:
    df_ibge = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("encoding", "UTF-8")
        .options(**adls_options)
        .load(IBGE_BRONZE_PATH)
        .select(
            upper(trim(col("nome_municipio"))).alias("cidade_ibge"),
            col("sigla_uf").alias("uf_ibge"),
            expr("TRY_CAST(TRY_CAST(populacao AS DOUBLE) AS BIGINT)").alias("populacao_cidade"),
            # Regra oficial (Negócio 7): enriquecimento IBGE inclui renda,
            # não só população. Coluna gerada pelo 00_setup_ibge (tabela
            # Sidra 10295 — renda domiciliar per capita por município).
            expr("TRY_CAST(renda_media_per_capita AS DOUBLE)").alias("renda_media_per_capita"),
        )
    )
    ibge_disponivel = True
    print(f"[OK] IBGE: {df_ibge.count()} municípios carregados.")
except Exception as e:
    ibge_disponivel = False
    print(
        f"[AVISO] Arquivo IBGE não encontrado em '{IBGE_BRONZE_PATH}'. "
        f"O enriquecimento de população/renda será ignorado. "
        f"Execute o notebook 'utils/00_setup_ibge' para gerar esse arquivo. "
        f"Erro: {e}"
    )

## Preparação da base para os KPIs

O id_loja não está diretamente nos itens — ele vem da tabela
vendas_caixa (cada transação pertence a uma loja).
Fazemos o JOIN: itens -> vendas_caixa -> lojas.

In [0]:
# Buscar id_loja e dt_venda da Bronze de vendas_caixa
# (fonte correta — id_loja nos itens pode estar nulo)
df_vendas_dim = (
    read_delta(BRONZE_VENDAS_CAIXA_PATH, adls_options)
    .select(
        col("id_transacao"),
        expr("TRY_CAST(id_loja AS BIGINT)").alias("id_loja"),
        col("dt_venda"),
    )
    .dropDuplicates(["id_transacao"])
)

print(f"Transações disponíveis: {df_vendas_dim.count():,}")

df_base = (
    df_itens
    .select(
        "id_transacao", "id_item_venda",
        "valor_item_analitico", "codigo_barras_produto",
        "flag_codigo_barras_ausente", "categoria_produto",
    )
    .join(df_vendas_dim, on="id_transacao", how="inner")
    .join(
        df_lojas.select(
            "id_loja", "nome_loja", "cidade_loja",
            "estado_tratado", "flag_cnpj_invalido", "flag_uf_invalido"
        ),
        on="id_loja",
        how="inner",
    )
    .withColumn("ano",      year(col("dt_venda").cast("date")))
    .withColumn("mes",      month(col("dt_venda").cast("date")))
    .withColumn("semestre",
        when(col("mes") <= 6, lit(1)).otherwise(lit(2))
    )
)

print(f"df_base: {df_base.count():,} linhas")

## KPI 4 — Receita total por loja por ano

In [0]:
df_kpi_receita_ano = (
    df_base
    .groupBy("id_loja", "nome_loja", "cidade_loja", "estado_tratado", "ano")
    .agg(
        spark_round(spark_sum("valor_item_analitico"), 2).alias("receita_total"),
        countDistinct("id_transacao").alias("qtd_transacoes"),
    )
)

print(f"KPI 4 — Receita anual: {df_kpi_receita_ano.count()} linha(s).")
display(df_kpi_receita_ano.orderBy("id_loja", "ano").limit(10))

####Ano completo vs. parcial

Um ano é considerado completo para uma loja quando existem 12 meses distintos de dados de venda naquele ano. Isso cobre tanto anos parciais no início da série (ex.: 2024, com dados só a partir de novembro) quanto o ano corrente ainda em andamento (ex.: 2026). A flag ano_completo deve ser usada no Looker para decidir automaticamente se um card exibe o valor normal ou uma versão com aviso/anualizada — sem depender de nota manual.



In [0]:
df_meses_por_ano = (
    df_base
    .groupBy("id_loja", "ano")
    .agg(countDistinct("mes").alias("qtd_meses_com_dados"))
    .withColumn("ano_completo", col("qtd_meses_com_dados") == 12)
)

print("[Frente A] Anos parciais detectados:")
display(
    df_meses_por_ano
    .filter(~col("ano_completo"))
    .orderBy("id_loja", "ano")
)

## KPI 5 — Crescimento YoY da receita por loja

Usa função de janela `LAG()` sobre o KPI 4 para comparar a receita
do ano atual com o ano anterior, por loja.

In [0]:
janela_yoy = Window.partitionBy("id_loja").orderBy("ano")

df_kpi_yoy = (
    df_kpi_receita_ano
    .join(
        df_meses_por_ano.select("id_loja", "ano", "ano_completo"),
        on=["id_loja", "ano"],
        how="left",
    )
    .withColumn(
        "receita_ano_anterior",
        lag("receita_total", 1).over(janela_yoy)
    )
    .withColumn(
        "ano_anterior_completo",
        lag("ano_completo", 1).over(janela_yoy)
    )
    .withColumn(
        "crescimento_yoy_pct",
        when(
            col("receita_ano_anterior").isNotNull()
            & (col("receita_ano_anterior") > 0)
            # Frente A: só calcula YoY se AMBOS os anos comparados forem
            # completos — evita comparar ano cheio com ano parcial
            # (causa raiz do YoY caindo para negativo em 2026).
            & (col("ano_completo") == True)
            & (col("ano_anterior_completo") == True),
            spark_round(
                (col("receita_total") - col("receita_ano_anterior"))
                / col("receita_ano_anterior") * 100,
                2
            )
        ).otherwise(lit(None))
    )
)

print(f"KPI 5 — YoY: {df_kpi_yoy.count()} linha(s).")
display(df_kpi_yoy.orderBy("id_loja", "ano").limit(10))

## KPI 6 — Número de transações por loja por mês

In [0]:
df_kpi_transacoes_mes = (
    df_base
    .groupBy("id_loja", "nome_loja", "cidade_loja", "estado_tratado", "ano", "mes")
    .agg(
        countDistinct("id_transacao").alias("qtd_transacoes"),
        spark_round(spark_sum("valor_item_analitico"), 2).alias("receita_mes"),
    )
)

print(f"KPI 6 — Transações/mês: {df_kpi_transacoes_mes.count()} linha(s).")
display(df_kpi_transacoes_mes.orderBy("id_loja", "ano", "mes").limit(10))

####Comparações mês a mês seguras com anos parciais

Com 2024 e 2026 parciais, comparar o ano inteiro distorce a leitura (é o que a coluna ano_completo já resolve para o YoY anual). Para cards de acompanhamento mês a mês no Looker, dois cálculos adicionais são seguros mesmo com anos parciais, porque nenhum deles depende do ano estar completo — cada um compara o mesmo recorte de tempo nos dois períodos:

 - crescimento_mom_pct: mês vs. mês imediatamente anterior (ex.: fev/2025 vs. jan/2025). Sempre válido, independe do ano.
 - crescimento_mesmo_mes_ano_anterior_pct: o mesmo mês comparado entre dois anos (ex.: março/2026 vs. março/2025). Só exige que aquele mês específico exista nos dois anos — não exige o ano inteiro.

Nenhum dos dois deve ser confundido com o YoY anual (crescimento_yoy_pct), que segue exigindo ano_completo = True nos dois anos.

In [0]:
janela_mom = Window.partitionBy("id_loja").orderBy("ano", "mes")
janela_mesmo_mes = Window.partitionBy("id_loja", "mes").orderBy("ano")

df_kpi_comparacoes_mensais = (
    df_kpi_transacoes_mes
    .withColumn(
        "receita_mes_anterior",
        lag("receita_mes", 1).over(janela_mom)
    )
    .withColumn(
        "crescimento_mom_pct",
        when(
            col("receita_mes_anterior").isNotNull() & (col("receita_mes_anterior") > 0),
            spark_round(
                (col("receita_mes") - col("receita_mes_anterior")) / col("receita_mes_anterior") * 100,
                2,
            )
        ).otherwise(lit(None))
    )
    .withColumn(
        "receita_mesmo_mes_ano_anterior",
        lag("receita_mes", 1).over(janela_mesmo_mes)
    )
    .withColumn(
        "crescimento_mesmo_mes_ano_anterior_pct",
        when(
            col("receita_mesmo_mes_ano_anterior").isNotNull() & (col("receita_mesmo_mes_ano_anterior") > 0),
            spark_round(
                (col("receita_mes") - col("receita_mesmo_mes_ano_anterior"))
                / col("receita_mesmo_mes_ano_anterior") * 100,
                2,
            )
        ).otherwise(lit(None))
    )
    .select(
        "id_loja", "ano", "mes",
        "crescimento_mom_pct",
        "crescimento_mesmo_mes_ano_anterior_pct",
    )
)

print(f"Comparações mês a mês: {df_kpi_comparacoes_mensais.count()} linha(s).")
display(df_kpi_comparacoes_mensais.orderBy("id_loja", "ano", "mes").limit(15))

## KPI 7 — Enriquecimento IBGE (população por cidade)

JOIN entre as lojas da Silver e o arquivo de municípios do IBGE,
usando `cidade_loja` (normalizada na Silver) como chave de JOIN.
Lojas sem correspondência no IBGE recebem `populacao_cidade = null`.

In [0]:
# Dicionário de correção: distritos/bairros que não aparecem no IBGE
# mapeados para o município oficial ao qual pertencem
CIDADE_CORRECTION_MAP = {
    "PORTO DAS CAIXAS": "ITABORAÍ",
    "RIOGRANDINA":      "NOVA FRIBURGO",
    "CAMPOS ELYSEOS":   "DUQUE DE CAXIAS",
    "BARRA DE SÃO JOÃO": "CASIMIRO DE ABREU",
    "CUNHAMBEBE":       "ANGRA DOS REIS",
}

def corrigir_cidade(nome):
    if nome is None:
        return nome
    return CIDADE_CORRECTION_MAP.get(nome.upper().strip(), nome.upper().strip())

corrigir_cidade_udf = udf(corrigir_cidade, StringType())

df_lojas_enriquecidas = df_lojas.select(
    "id_loja", "nome_loja", "cidade_loja", "estado_tratado",
    "cnpj_tratado", "flag_cnpj_invalido", "flag_uf_invalido"
)

if ibge_disponivel:
    df_lojas_enriquecidas = (
        df_lojas_enriquecidas
        .withColumn("cidade_upper", corrigir_cidade_udf(col("cidade_loja")))
        .join(
            df_ibge.withColumn("_ibge_existe", lit(True)),
            on=(
                (col("cidade_upper") == col("cidade_ibge")) &
                (col("estado_tratado") == col("uf_ibge"))
            ),
            how="left",
        )
        .withColumn("flag_sem_match_ibge", col("_ibge_existe").isNull())
        .drop("_ibge_existe", "cidade_upper", "cidade_ibge", "uf_ibge")
    )

    qtd_sem_match = df_lojas_enriquecidas.filter(col("flag_sem_match_ibge")).count()
    print(f"[KPI 7] {qtd_sem_match} loja(s) sem correspondência no IBGE.")
else:
    df_lojas_enriquecidas = (
        df_lojas_enriquecidas
        .withColumn("populacao_cidade", lit(None).cast("long"))
        .withColumn("renda_media_per_capita", lit(None).cast("double"))
        .withColumn("flag_sem_match_ibge", lit(True))
    )
    print("[KPI 7] Enriquecimento IBGE ignorado (arquivo não disponível).")

display(df_lojas_enriquecidas.limit(10))

## KPI 8 — Ticket médio por loja por semestre

In [0]:
df_kpi_ticket_semestre = (
    df_base
    .groupBy("id_loja", "nome_loja", "cidade_loja", "estado_tratado", "ano", "semestre")
    .agg(
        spark_round(spark_sum("valor_item_analitico"), 2).alias("receita_total"),
        countDistinct("id_transacao").alias("qtd_transacoes"),
    )
    .withColumn(
        "ticket_medio",
        when(
            col("qtd_transacoes") > 0,
            spark_round(col("receita_total") / col("qtd_transacoes"), 2)
        ).otherwise(lit(None))
    )
)

print(f"KPI 8 — Ticket médio/semestre: {df_kpi_ticket_semestre.count()} linha(s).")
display(df_kpi_ticket_semestre.orderBy("id_loja", "ano", "semestre").limit(10))

## Montagem da tabela Gold final

A tabela Gold consolida os KPIs calculados em uma estrutura
única, no nível de granularidade mais detalhado (loja + ano + mês),
com os atributos de enriquecimento do IBGE e os KPIs de semestre
e YoY desnormalizados via JOIN.

In [0]:
df_gold = (
    df_kpi_transacoes_mes
    .withColumn(
        "semestre",
        when(col("mes") <= 6, lit(1)).otherwise(lit(2))
    )
    .join(
        df_kpi_yoy.select(
            "id_loja", "ano",
            "receita_total", "receita_ano_anterior", "crescimento_yoy_pct",
            "ano_completo",
        ),
        on=["id_loja", "ano"],
        how="left",
    )
    .join(
        df_kpi_ticket_semestre.select(
            "id_loja", "ano", "semestre", "ticket_medio"
        ),
        on=["id_loja", "ano", "semestre"],
        how="left",
    )
    .join(
        # Extensão MoM: comparações mês a mês seguras mesmo com anos
        # parciais (não exigem ano_completo, ao contrário do YoY anual).
        df_kpi_comparacoes_mensais,
        on=["id_loja", "ano", "mes"],
        how="left",
    )
    .join(
        df_lojas_enriquecidas.select(
            "id_loja", "cnpj_tratado", "populacao_cidade",
            "renda_media_per_capita",
            "flag_cnpj_invalido", "flag_uf_invalido", "flag_sem_match_ibge"
        ),
        on="id_loja",
        how="left",
    )
    .select(
        "id_loja", "nome_loja", "cidade_loja", "estado_tratado",
        "cnpj_tratado", "populacao_cidade", "renda_media_per_capita",
        "ano", "mes", "semestre",
        "receita_mes", "qtd_transacoes",
        "receita_total", "receita_ano_anterior", "crescimento_yoy_pct",
        "ano_completo",
        "crescimento_mom_pct",
        "crescimento_mesmo_mes_ano_anterior_pct",
        "ticket_medio",
        "flag_cnpj_invalido", "flag_uf_invalido", "flag_sem_match_ibge",
    )
)

print(f"Gold final: {df_gold.count()} linha(s).")

# Garantir tipos consistentes com o Delta existente antes de gravar
df_gold = df_gold.withColumn("id_loja", col("id_loja").cast("long"))

display(df_gold.limit(10))

## KPI 10 — Validação: todas as lojas da Silver estão na Gold

Confirma que nenhuma loja foi perdida no caminho Silver -> Gold.
Se alguma loja existir na Silver mas não na Gold, o pipeline
interrompe com erro — isso indica um problema no JOIN ou nos
filtros de qualidade desta camada.

In [0]:
ids_silver = set([str(row["id_loja"]) for row in df_lojas.select("id_loja").collect()])
ids_gold   = set([str(row["id_loja"]) for row in df_gold.select("id_loja").distinct().collect()])

lojas_faltando = ids_silver - ids_gold

if lojas_faltando:
    raise RuntimeError(
        f"ERRO KPI 10: {len(lojas_faltando)} loja(s) da Silver não estão "
        f"na Gold: {lojas_faltando}. Verifique os JOINs e filtros desta camada."
    )

print(f"[KPI 10] Validação OK: todas as {len(ids_silver)} loja(s) da Silver "
      f"estão presentes na Gold.")

## Verificação de pré-condição

In [0]:
verificar_destino_limpo(
    GOLD_LOJAS_PATH,
    adls_options,
    permitir_existente=(GOLD_WRITE_MODE == "overwrite"),
)

## Escrita no Delta (Gold) e no SQL Server

A Gold é a **única camada** que grava no SQL Server. O Delta serve
de fonte para os notebooks de Analysis (gráficos de KPI).

In [0]:
(
    df_gold.write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(GOLD_WRITE_MODE)
    .save(GOLD_LOJAS_PATH)
)

print(f"[OK] Delta Gold gravado em '{GOLD_LOJAS_PATH}'.")


In [0]:
write_sql_table_typed(df_gold, SQL_TABLE_GOLD_LOJAS)